# Modelo_GS v3 — Motor de recomendação do painel (camada 3 da IA)

**Pergunta:** em quais distritos há mais gente que provavelmente tem direito e não recebe?

**Ideia do modelo:** distritos com perfil social parecido deveriam ter cobertura parecida
de BPC e Bolsa Família. Treinamos um modelo que prevê a cobertura *esperada* a partir
só de variáveis de **necessidade** (pobreza, idade, densidade do CadÚnico). A diferença
entre o esperado e o observado é a **lacuna estimada de acesso**. Barreiras de acesso
(distância ao CRAS, risco) ficam **fora** do modelo de propósito: depois testamos se
elas explicam a lacuna, e usamos no índice de prioridade.

**Só dados públicos reais** (o edital não aceita dados fictícios no painel):
| Tabela | Fonte | Nível |
|---|---|---|
| Famílias no CadÚnico (total e por faixa de renda, se houver) | Portal de Dados Abertos SP / SMADS | distrito |
| Famílias beneficiárias do Bolsa Família | Portal de Dados Abertos SP / SMADS | distrito |
| Beneficiários do BPC (idoso e PcD) | Portal de Dados Abertos SP / SMADS | distrito |
| População total, 0–6 e 65+ (Censo 2022) | TabNet SMS-SP / IBGE | distrito |
| Distância média ao CRAS | notebook de tratamento do grupo (GeoSampa) | distrito |
| Setores de risco R3/R4 | notebook de tratamento do grupo (GeoSampa) | distrito |

Cada saída leva a coluna `origem` = "REAL" ou "ESTIMATIVA DO MODELO".

**Saídas** (`saida_modelo/`, CSV com `;` e vírgula decimal para o Power BI): `fato_indicadores_distrito`, `fato_lacuna_programa` (distrito × programa), `dim_programa` (os 7 programas do app), `fato_recomendacoes` (com programa e CRAS de referência), `fato_comparacao_risco`, `fato_insights`, além de métricas, teste de barreiras e sensibilidade.

In [10]:

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from google.colab import drive

# ------------------------------------------------------------------
# CONEXÃO COM GOOGLE DRIVE
# ------------------------------------------------------------------
drive.mount('/content/drive')

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

# ------------------------------------------------------------------
# CONFIGURAÇÃO DE CAMINHOS
# ------------------------------------------------------------------
PASTA = Path('/content/drive/MyDrive/Dados')

PASTA_ENTRADA = PASTA / 'saida'
PASTA_SAIDA = PASTA / 'saida_modelo'

# Cria a pasta de saída caso ainda não exista
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

ARQUIVO_BASE = PASTA_ENTRADA / 'distritos_base.csv'

SEMENTE = 42

# ------------------------------------------------------------------
# CONFIGURAÇÃO DAS COLUNAS
# ------------------------------------------------------------------
COLUNAS = {
    "cod_distrito": "cod_distrito",
    "distrito": "distrito",
    "subprefeitura": "subprefeitura",
    "pop_total": "pop_total",
    "pop_0_6": "pop_0_6",
    "pop_65_mais": "pop_65_mais",
    "familias_cadunico": "familias_cadunico",
    "familias_cadunico_pobreza": "familias_cadunico_pobreza",
    "familias_pbf": "familias_pbf",
    "bpc_idoso": "bpc_idoso",
    "bpc_pcd": "bpc_pcd",
    "dist_cras_km": "dist_cras_km",
    "setores_risco_r3r4": "setores_risco_r3r4",
}

# ------------------------------------------------------------------
# CONFIGURAÇÃO DO MODELO
# ------------------------------------------------------------------
PESOS = {
    "lacuna_pessoas": 0.45,
    "dist_cras_km": 0.25,
    "setores_risco_r3r4": 0.15,
    "cadunico_por_1000": 0.15
}

CORTES = {
    "urgente": 0.90,
    "alta": 0.70,
    "média": 0.40
}


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
# ------------------------------------------------------------------
# LOCALIZA O distritos_base.csv (evita o FileNotFoundError)
# ------------------------------------------------------------------
if not ARQUIVO_BASE.exists():
    print(f"Não achei {ARQUIVO_BASE}")
    print("Conteúdo de", PASTA_ENTRADA, ":", [p.name for p in PASTA_ENTRADA.glob('*')] if PASTA_ENTRADA.exists() else "PASTA NÃO EXISTE")
    achados = list(Path('/content/drive/MyDrive').rglob('distritos_base*.csv'))
    if achados:
        ARQUIVO_BASE = achados[0]
        print("Usando o arquivo encontrado em:", ARQUIVO_BASE)
    else:
        print("O arquivo não está no Drive. Rode o notebook de tratamento até o fim (ele gera o CSV) ou faça upload agora:")
        from google.colab import files
        enviado = files.upload()
        ARQUIVO_BASE = Path('/content') / next(iter(enviado))
print("Arquivo base:", ARQUIVO_BASE)


Arquivo base: /content/drive/MyDrive/Dados/saida/distritos_base.csv


In [12]:
def carregar(caminho: Path | None = None) -> pd.DataFrame:
    df = pd.read_csv(caminho or ARQUIVO_BASE)
    inverso = {v: k for k, v in COLUNAS.items() if v in df.columns}
    df = df.rename(columns=inverso)
    obrigatorias = ["distrito", "subprefeitura", "pop_total", "pop_65_mais", "familias_cadunico", "familias_pbf", "bpc_idoso"]
    faltando = [c for c in obrigatorias if c not in df.columns]
    if faltando:
        raise KeyError(f"Colunas obrigatórias ausentes: {faltando}. Ajuste o dicionário COLUNAS.")
    for c in ["pop_0_6", "bpc_pcd", "dist_cras_km", "setores_risco_r3r4", "familias_cadunico_pobreza"]:
        if c not in df.columns:
            df[c] = np.nan
    return df


def indicadores(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    d["cadunico_por_1000"] = d.familias_cadunico / d.pop_total * 1000
    d["pct_0_6"] = d.pop_0_6 / d.pop_total
    d["pct_65"] = d.pop_65_mais / d.pop_total
    tem_pobreza = d.familias_cadunico_pobreza.notna().all()
    d["pct_pobreza_cadunico"] = d.familias_cadunico_pobreza / d.familias_cadunico if tem_pobreza else np.nan
    # Coberturas observadas (REAL)
    base_pbf = d.familias_cadunico_pobreza if tem_pobreza else d.familias_cadunico
    d["cobertura_pbf"] = d.familias_pbf / base_pbf
    d["taxa_bpc_idoso"] = d.bpc_idoso / d.pop_65_mais
    # divisão por zero vira inf e quebra o sklearn
    d = d.replace([np.inf, -np.inf], np.nan)
    return d

## Modelo: cobertura esperada a partir da necessidade
Variáveis de necessidade (entram): CadÚnico por 1.000 hab., % 0–6, % 65+, % em pobreza no CadÚnico.
Barreiras (não entram): distância ao CRAS e risco.
Validação cruzada 5-fold repetida; a previsão de cada distrito vem de um modelo que **não** o viu no treino.

In [13]:
def features_necessidade(d: pd.DataFrame) -> list[str]:
    cand = ["cadunico_por_1000", "pct_0_6", "pct_65", "pct_pobreza_cadunico"]
    return [c for c in cand if d[c].notna().all()]


def comparar_modelos(X: pd.DataFrame, y: pd.Series, alvo: str):
    modelos = {
        "Regressão linear (Ridge)": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 13))),
        "Random Forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=SEMENTE),
        "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, max_depth=2, learning_rate=0.05, random_state=SEMENTE),
    }
    linhas = []
    for nome, m in modelos.items():
        maes, r2s = [], []
        for rep in range(5):
            kf = KFold(5, shuffle=True, random_state=SEMENTE + rep)
            p = cross_val_predict(m, X, y, cv=kf)
            maes.append(mean_absolute_error(y, p))
            r2s.append(r2_score(y, p))
        linhas.append({"alvo": alvo, "modelo": nome, "MAE_cv": np.mean(maes), "R2_cv": np.mean(r2s), "R2_cv_desvio": np.std(r2s)})
    res = pd.DataFrame(linhas).sort_values("MAE_cv")
    return res, res.iloc[0]["modelo"], modelos[res.iloc[0]["modelo"]]


def prever_esperado(d: pd.DataFrame, alvo: str) -> tuple[pd.Series, pd.DataFrame]:
    X, y = d[features_necessidade(d)], d[alvo]
    res, nome, modelo = comparar_modelos(X, y, alvo)
    esperado = pd.Series(cross_val_predict(modelo, X, y, cv=KFold(5, shuffle=True, random_state=SEMENTE)), index=d.index)
    res["escolhido"] = res.modelo == nome
    r2 = res.loc[res.escolhido, "R2_cv"].iloc[0]
    if r2 < 0.1:
        print(f"⚠️  {alvo}: o melhor modelo tem R² de validação {r2:.2f}, ou seja, quase não supera a média da cidade. "
              "A 'cobertura esperada' fica perto da média; digam isso na banca em vez de esconder.")
    return esperado.clip(lower=0), res

In [14]:
def lacunas(d: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    d = d.copy()
    metricas = []
    d["bpc_idoso_esperado_taxa"], m1 = prever_esperado(d, "taxa_bpc_idoso")
    d["cobertura_pbf_esperada"], m2 = prever_esperado(d, "cobertura_pbf")
    metricas += [m1, m2]
    base_pbf = d.familias_cadunico_pobreza if d.familias_cadunico_pobreza.notna().all() else d.familias_cadunico
    # Lacuna = esperado - observado (só a parte positiva), em pessoas/famílias
    d["lacuna_bpc_idosos"] = ((d.bpc_idoso_esperado_taxa - d.taxa_bpc_idoso).clip(lower=0) * d.pop_65_mais).round()
    d["lacuna_pbf_familias"] = ((d.cobertura_pbf_esperada - d.cobertura_pbf).clip(lower=0) * base_pbf).round()
    # BPC é por pessoa e PBF por família: para somar, convertemos famílias em pessoas pelo tamanho médio (aprox. 2,5 em SP — ajuste se tiverem o dado)
    d["lacuna_pessoas"] = d.lacuna_bpc_idosos + d.lacuna_pbf_familias * 2.5
    return d, pd.concat(metricas, ignore_index=True)

## As barreiras explicam a lacuna?
Correlação de Spearman entre a lacuna relativa e distância ao CRAS / risco.
Se for positiva e significativa, sustenta a história do pitch: "a distância afasta as pessoas dos direitos".

**Com risco × sem risco** (bloco de zona de risco do painel) e **frases de insight**: mesmas contas das medidas DAX, com teste de Mann-Whitney. As frases só dizem que o risco "piora" quando a diferença é significativa.

In [15]:
def testar_barreiras(d: pd.DataFrame) -> pd.DataFrame:
    d = d.assign(lacuna_por_1000=d.lacuna_pessoas / d.pop_total * 1000)
    linhas = []
    for barreira in ["dist_cras_km", "setores_risco_r3r4"]:
        if d[barreira].notna().sum() > 10:
            rho, p = spearmanr(d[barreira], d.lacuna_por_1000, nan_policy="omit")
            linhas.append({"barreira": barreira, "spearman_rho": round(rho, 3), "p_valor": round(p, 4),
                           "leitura": ("associação positiva" if rho > 0 else "associação negativa") + (" (significativa a 5%)" if p < 0.05 else " (não significativa)")})
    return pd.DataFrame(linhas)

def comparar_risco(d: pd.DataFrame) -> pd.DataFrame:
    """Bloco 'zona de risco piora a situação': distritos com setores R3/R4 x sem.
    Mesmas contas das medidas DAX (razão de somas), com teste de Mann-Whitney entre os distritos dos dois grupos."""
    from scipy.stats import mannwhitneyu
    if d.setores_risco_r3r4.isna().all():
        return pd.DataFrame()
    d = d.assign(lacuna_por_1000=d.lacuna_pessoas / d.pop_total * 1000)
    com, sem = d[d.tem_risco == "Com R3/R4"], d[d.tem_risco == "Sem R3/R4"]
    base_pbf = "familias_cadunico_pobreza" if d.familias_cadunico_pobreza.notna().all() else "familias_cadunico"
    contas = {
        "Cobertura BPC idoso": (lambda g: g.bpc_idoso.sum() / g.pop_65_mais.sum(), "taxa_bpc_idoso", "%"),
        "Cobertura Bolsa Família": (lambda g: g.familias_pbf.sum() / g[base_pbf].sum(), "cobertura_pbf", "%"),
        "Distância média ao CRAS (km)": (lambda g: g.dist_cras_km.mean(), "dist_cras_km", "km"),
        "Pessoas sem benefício por 1.000 hab.": (lambda g: g.lacuna_pessoas.sum() / g.pop_total.sum() * 1000, "lacuna_por_1000", "por 1.000"),
    }
    linhas = []
    for nome, (f, col, unid) in contas.items():
        p = mannwhitneyu(com[col].dropna(), sem[col].dropna()).pvalue if len(com) > 2 and len(sem) > 2 else np.nan
        linhas.append({"indicador": nome, "unidade": unid, "com_r3r4": f(com), "sem_r3r4": f(sem),
                       "diferenca": f(com) - f(sem), "p_valor": round(p, 4), "n_com": len(com), "n_sem": len(sem),
                       "significativa_5pct": bool(p < 0.05) if pd.notna(p) else False,
                       "origem": "ESTIMATIVA DO MODELO" if col == "lacuna_por_1000" else "REAL"})
    return pd.DataFrame(linhas)


def gerar_insights(d: pd.DataFrame, barreiras: pd.DataFrame, comp: pd.DataFrame) -> pd.DataFrame:
    """Frases para o bloco Insights, em ordem de importância (no Power BI, mostrem as 4 primeiras).
    Regras: a direção ('maior'/'menor') vem do dado, nunca fixa; só se diz que algo 'agrava' quando o teste é significativo a 5%."""
    br = lambda x, casas=1: f"{x:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")  # noqa: E731
    def leitura(p):
        return (" A diferença é significativa a 5%." if p < 0.05
                else f" A diferença não é significativa (p = {br(p, 2)}).")
    frases = []
    b = barreiras.set_index("barreira") if len(barreiras) else pd.DataFrame()
    r = comp.set_index("indicador") if len(comp) else pd.DataFrame()

    # 1. Risco x lacuna (contínuo: nº de setores R3/R4 x pessoas sem benefício por 1.000 hab.)
    risco_signif = False
    if "setores_risco_r3r4" in b.index:
        x = b.loc["setores_risco_r3r4"]
        if x.p_valor < 0.05 and x.spearman_rho > 0:
            risco_signif = True
            frases.append(f"Quanto mais setores de risco R3/R4 o distrito tem, mais gente com direito provável fica sem benefício "
                          f"por habitante (correlação de Spearman {br(x.spearman_rho, 2)}, significativa a 5%): a área de risco agrava a falta de acesso.")
        else:
            frases.append(f"O número de setores de risco R3/R4 não tem relação clara com a falta de benefício "
                          f"(Spearman {br(x.spearman_rho, 2)}, p = {br(x.p_valor, 2)}).")

    # 2. Concentração nos urgentes
    urg = d[d.classe_prioridade == "urgente"]
    if len(urg) and d.lacuna_pessoas.sum() > 0:
        frases.append(f"Os {len(urg)} distritos urgentes concentram {br(urg.lacuna_pessoas.sum() / d.lacuna_pessoas.sum() * 100, 0)}% "
                      f"das pessoas com direito provável sem benefício (cerca de {br(urg.lacuna_pessoas.sum(), 0)} pessoas).")

    if len(r):
        # 3. Bolsa Família com x sem risco
        p = r.loc["Cobertura Bolsa Família"]
        dif = abs(p.diferenca) * 100
        frases.append(f"Nos distritos com área R3/R4, {br(p.com_r3r4 * 100)}% das famílias do CadÚnico recebem Bolsa Família, "
                      f"contra {br(p.sem_r3r4 * 100)}% nos sem risco ({br(dif)} pontos {'a menos' if p.diferenca < 0 else 'a mais'})."
                      + leitura(p.p_valor))
        # 4. BPC: a comparação bruta engana (distritos com risco são mais pobres)
        q = r.loc["Cobertura BPC idoso"]
        if q.diferenca > 0:
            frases.append(f"A cobertura de BPC idoso é maior nos distritos com R3/R4 ({br(q.com_r3r4 * 100)}% contra {br(q.sem_r3r4 * 100)}%) "
                          "porque eles têm mais famílias pobres (mais CadÚnico por habitante). Por isso o modelo compara cada distrito com os de perfil parecido, e não com a média da cidade.")
        else:
            frases.append(f"A cobertura de BPC idoso é menor nos distritos com R3/R4 ({br(q.com_r3r4 * 100)}% contra {br(q.sem_r3r4 * 100)}%)."
                          + leitura(q.p_valor))
        # só usa o teste binário da lacuna se o contínuo (frase 1) não foi significativo, para não se contradizer
        if not risco_signif:
            l = r.loc["Pessoas sem benefício por 1.000 hab."]
            frases.append(f"Nos distritos com R3/R4 há {br(l.com_r3r4, 0)} pessoas com direito provável sem benefício a cada 1.000 habitantes, "
                          f"contra {br(l.sem_r3r4, 0)} nos sem risco." + leitura(l.p_valor))

    # 5. Distância ao CRAS
    if "dist_cras_km" in b.index:
        x = b.loc["dist_cras_km"]
        if x.p_valor < 0.05:
            frases.append(("Quanto mais longe do CRAS, mais" if x.spearman_rho > 0 else "Quanto mais longe do CRAS, menos")
                          + f" gente sem benefício por habitante (Spearman {br(x.spearman_rho, 2)}, significativa a 5%).")
        else:
            frases.append(f"A distância ao CRAS não tem relação clara com a falta de benefício "
                          f"(Spearman {br(x.spearman_rho, 2)}, p = {br(x.p_valor, 2)}).")
    return pd.DataFrame({"ordem": range(1, len(frases) + 1), "insight": frases, "origem": "ESTIMATIVA DO MODELO"})


## Índice de prioridade e recomendações
Normalização min-max (0–1) de cada componente, soma ponderada × 100.
Só distritos **urgente** ou **alta** recebem recomendação de campanha (decisão do grupo).

In [16]:
def normalizar(s: pd.Series) -> pd.Series:
    if s.isna().all():  # coluna sem dado (ex.: barreira ausente) não pode zerar o índice com NaN
        return pd.Series(0.0, index=s.index)
    s = s.fillna(s.median())
    return (s - s.min()) / (s.max() - s.min()) if s.max() > s.min() else s * 0


def indice_prioridade(d: pd.DataFrame, pesos: dict = PESOS) -> pd.Series:
    return sum(normalizar(d[c]) * w for c, w in pesos.items()) * 100


def classificar(indice: pd.Series) -> pd.Series:
    q = {k: indice.quantile(v) for k, v in CORTES.items()}
    return pd.Series(np.select([indice >= q["urgente"], indice >= q["alta"], indice >= q["média"]],
                               ["urgente", "alta", "média"], "baixa"), index=indice.index)


def sensibilidade(d: pd.DataFrame, n: int = 200) -> pd.DataFrame:
    """Sorteia pesos ±50% e mede quanto o ranking muda (Spearman) e quantos distritos 'urgente' se mantêm."""
    rng = np.random.default_rng(SEMENTE)
    base = indice_prioridade(d)
    top_base = set(d.index[classificar(base) == "urgente"])
    linhas = []
    for _ in range(n):
        w = {k: v * rng.uniform(0.5, 1.5) for k, v in PESOS.items()}
        tot = sum(w.values())
        w = {k: v / tot for k, v in w.items()}
        alt = indice_prioridade(d, w)
        top_alt = set(d.index[classificar(alt) == "urgente"])
        linhas.append({"spearman_ranking": spearmanr(base, alt)[0], "urgentes_mantidos_pct": len(top_base & top_alt) / max(len(top_base), 1)})
    return pd.DataFrame(linhas).describe().loc[["mean", "min", "25%"]].round(3)


# Mesmos 7 programas do simulador do app (simulador/regras.csv): o painel e o app falam a mesma língua.
DIM_PROGRAMA = pd.DataFrame([
    (1, "Documentação básica", "Cartório / Poupatempo", False),
    (2, "CadÚnico", "CRAS (SMADS)", True),
    (3, "Bolsa Família", "MDS via CadÚnico", True),
    (4, "BPC", "INSS", True),
    (5, "Tarifa Social de Energia", "Distribuidora (Enel SP)", False),
    (6, "Pé-de-Meia", "MEC", False),
    (7, "Farmácia Popular", "Ministério da Saúde", False),
], columns=["id_programa", "programa", "onde_pedir", "tem_dado_publico_por_distrito"])


def lacuna_por_programa(d: pd.DataFrame) -> pd.DataFrame:
    """Formato longo (distrito x programa) para ligar à dim_programa: o mesmo eixo que o app usa no Plano de Direitos."""
    base_pbf = d.familias_cadunico_pobreza if d.familias_cadunico_pobreza.notna().all() else d.familias_cadunico
    bpc = pd.DataFrame({"cod_distrito": d.cod_distrito, "programa": "BPC", "publico": "idosos 65+",
                        "publico_base": d.pop_65_mais, "beneficiarios": d.bpc_idoso,
                        "cobertura_real": d.taxa_bpc_idoso, "cobertura_esperada": d.bpc_idoso_esperado_taxa,
                        "lacuna": d.lacuna_bpc_idosos, "unidade": "pessoas", "lacuna_pessoas": d.lacuna_bpc_idosos})
    pbf = pd.DataFrame({"cod_distrito": d.cod_distrito, "programa": "Bolsa Família", "publico": "famílias no CadÚnico",
                        "publico_base": base_pbf, "beneficiarios": d.familias_pbf,
                        "cobertura_real": d.cobertura_pbf, "cobertura_esperada": d.cobertura_pbf_esperada,
                        "lacuna": d.lacuna_pbf_familias, "unidade": "famílias", "lacuna_pessoas": d.lacuna_pbf_familias * 2.5})
    out = pd.concat([bpc, pbf], ignore_index=True).merge(DIM_PROGRAMA[["id_programa", "programa"]], on="programa")
    out["origem"] = "ESTIMATIVA DO MODELO"
    return out


def recomendar(d: pd.DataFrame) -> pd.DataFrame:
    p75 = lambda c: d[c].quantile(0.75)  # noqa: E731
    pct = lambda x: f"{x:.1%}".replace(".", ",")  # noqa: E731  (vírgula decimal no painel)
    recs = []
    for _, r in d[d.classe_prioridade.isin(["urgente", "alta"])].iterrows():
        def add(acao, programa, motivo, publico):
            recs.append({"cod_distrito": r.get("cod_distrito"), "distrito": r.distrito, "subprefeitura": r.subprefeitura,
                         "classe_prioridade": r.classe_prioridade, "acao": acao, "programa": programa,
                         "cras_referencia": r.get("cras_referencia"), "motivo": motivo, "publico_estimado": publico,
                         "origem": "ESTIMATIVA DO MODELO"})
        if r.lacuna_bpc_idosos >= p75("lacuna_bpc_idosos") and r.lacuna_bpc_idosos > 0:
            add("Mutirão BPC: busca ativa de idosos 65+ com ACS e CRAS", "BPC",
                f"Cobertura de BPC idoso {pct(r.taxa_bpc_idoso)} vs esperada {pct(r.bpc_idoso_esperado_taxa)}", int(r.lacuna_bpc_idosos))
        if r.lacuna_pbf_familias >= p75("lacuna_pbf_familias") and r.lacuna_pbf_familias > 0:
            add("Busca ativa Bolsa Família e atualização do CadÚnico", "Bolsa Família",
                f"Cobertura do PBF {pct(r.cobertura_pbf)} vs esperada {pct(r.cobertura_pbf_esperada)}", int(r.lacuna_pbf_familias))
        if pd.notna(r.dist_cras_km) and r.dist_cras_km >= p75("dist_cras_km"):
            add("CRAS itinerante ou atendimento do CadÚnico dentro da UBS", "CadÚnico",
                f"Distância média ao CRAS de {r.dist_cras_km:.1f} km (entre as 25% maiores)".replace(".", ",", 1), int(r.familias_cadunico))
        if pd.notna(r.setores_risco_r3r4) and r.setores_risco_r3r4 >= max(1, p75("setores_risco_r3r4")):
            add("Priorizar visitas dos ACS nos setores de risco R3/R4", "Todos",
                f"{int(r.setores_risco_r3r4)} setores R3/R4 no distrito", None)
        if not any(x["distrito"] == r.distrito for x in recs):
            add("Mutirão de documentação e CadÚnico (certidão, CPF, CIN)", "Documentação básica",
                "Prioridade alta pelo conjunto de indicadores", None)
    return pd.DataFrame(recs)


In [17]:
def salvar_pbi(df: pd.DataFrame, caminho: Path, index: bool = False) -> None:
    """CSV para o Power BI em português: separador ';' e decimal ',' (mesmo padrão do tratamento)."""
    df.to_csv(caminho, sep=";", decimal=",", index=index, encoding="utf-8-sig")


def rodar(caminho: Path | None = None, saida: Path = PASTA_SAIDA) -> dict:
    saida.mkdir(exist_ok=True, parents=True)
    d = indicadores(carregar(caminho))
    # CRAS de referência de cada distrito (o mais próximo do centro), do tratamento: liga a campanha à unidade do app
    arq_acesso = PASTA_ENTRADA / "fato_acesso_distrito_servico.csv"
    if arq_acesso.exists() and "cod_distrito" in d:
        ac = pd.read_csv(arq_acesso, sep=";", decimal=",", encoding="utf-8-sig")
        ac = ac[ac.servico == "CRAS"].set_index("cod_distrito").servico_mais_proximo
        d["cras_referencia"] = d.cod_distrito.map(ac)
    else:
        d["cras_referencia"] = None
    d, metricas = lacunas(d)
    d["indice_prioridade"] = indice_prioridade(d).round(1)
    d["classe_prioridade"] = classificar(d.indice_prioridade)
    d["tem_risco"] = np.where(d.setores_risco_r3r4 > 0, "Com R3/R4", "Sem R3/R4")
    d["lacuna_por_1000"] = (d.lacuna_pessoas / d.pop_total * 1000).round(2)
    d["origem_indicadores"] = "REAL"
    d["origem_esperado_lacuna_indice"] = "ESTIMATIVA DO MODELO"
    barreiras = testar_barreiras(d)
    comp = comparar_risco(d)
    insights = gerar_insights(d, barreiras, comp)
    sens = sensibilidade(d)
    recs = recomendar(d)
    lac_prog = lacuna_por_programa(d)

    salvar_pbi(d, saida / "fato_indicadores_distrito.csv")
    salvar_pbi(recs, saida / "fato_recomendacoes.csv")
    salvar_pbi(lac_prog, saida / "fato_lacuna_programa.csv")
    salvar_pbi(DIM_PROGRAMA, saida / "dim_programa.csv")
    salvar_pbi(comp, saida / "fato_comparacao_risco.csv")
    salvar_pbi(insights, saida / "fato_insights.csv")
    salvar_pbi(metricas, saida / "metricas_modelo.csv")
    salvar_pbi(barreiras, saida / "teste_barreiras.csv")
    salvar_pbi(sens, saida / "sensibilidade_pesos.csv", index=True)

    print("== Modelos (validação cruzada) ==\n", metricas.round(4).to_string(index=False))
    print("\n== Barreiras x lacuna ==\n", barreiras.to_string(index=False))
    print("\n== Com risco R3/R4 x sem ==\n", comp.round(4).to_string(index=False))
    print("\n== Insights ==")
    for f in insights.insight:
        print(" -", f)
    print("\n== Sensibilidade dos pesos ==\n", sens.to_string())
    print("\n== Distritos por classe ==\n", d.classe_prioridade.value_counts().to_string())
    print("\n== Top 10 prioridade ==\n", d.nlargest(10, "indice_prioridade")[["distrito", "subprefeitura", "indice_prioridade", "classe_prioridade", "lacuna_bpc_idosos", "lacuna_pbf_familias"]].to_string(index=False))
    print(f"\nRecomendações geradas: {len(recs)} (só urgente/alta)")
    return {"distritos": d, "recomendacoes": recs, "metricas": metricas, "barreiras": barreiras,
            "comparacao_risco": comp, "insights": insights, "sensibilidade": sens,
            "lacuna_programa": lac_prog}


## Rodar
No Colab: monte o Drive ou faça upload de `saida/distritos_base.csv` e rode a célula abaixo. As saídas vão para `saida_modelo/` e entram no Power BI.

In [18]:
resultado = rodar()
resultado['distritos'].head()

== Modelos (validação cruzada) ==
           alvo                   modelo  MAE_cv   R2_cv  R2_cv_desvio  escolhido
taxa_bpc_idoso            Random Forest  0.0198  0.8119        0.0156       True
taxa_bpc_idoso        Gradient Boosting  0.0238  0.6718        0.0758      False
taxa_bpc_idoso Regressão linear (Ridge)  0.0244  0.6692        0.0163      False
 cobertura_pbf Regressão linear (Ridge)  0.0515  0.1910        0.0302       True
 cobertura_pbf            Random Forest  0.0552  0.1691        0.0286      False
 cobertura_pbf        Gradient Boosting  0.0599 -0.0419        0.1033      False

== Barreiras x lacuna ==
           barreira  spearman_rho  p_valor                                  leitura
      dist_cras_km         0.062   0.5479  associação positiva (não significativa)
setores_risco_r3r4         0.231   0.0238 associação positiva (significativa a 5%)

== Com risco R3/R4 x sem ==
                            indicador   unidade  com_r3r4  sem_r3r4  diferenca  p_valor  n_co

,cod_distrito,distrito,subprefeitura,pop_total,familias_cadunico,dist_cras_km,setores_risco_r3r4,familias_pbf,bpc_idoso,bpc_pcd,pop_65_mais,pop_0_6,familias_cadunico_pobreza,cadunico_por_1000,pct_0_6,pct_65,pct_pobreza_cadunico,cobertura_pbf,taxa_bpc_idoso,cras_referencia,bpc_idoso_esperado_taxa,cobertura_pbf_esperada,lacuna_bpc_idosos,lacuna_pbf_familias,lacuna_pessoas,indice_prioridade,classe_prioridade,tem_risco,lacuna_por_1000,origem_indicadores,origem_esperado_lacuna_indice
0,63,Pirituba,Pirituba/Jaraguá,179724.0,20449,0.709,7,6966,2308,1215,21916.0,NaN,NaN,113.780018,NaN,0.121943,NaN,0.340652,0.105311,"CRAS - Rua Guerino Giovani Leardini, 290",0.095215,0.359795,0.0,391.0,977.5,12.5,baixa,Com R3/R4,5.44,REAL,ESTIMATIVA DO MODELO
1,95,São Domingos,Pirituba/Jaraguá,88884.0,10745,2.776,3,3845,1011,560,11780.0,NaN,NaN,120.887899,NaN,0.132532,NaN,0.357841,0.085823,"CRAS - Rua Guerino Giovani Leardini, 290",0.109439,0.359726,278.0,20.0,328.0,18.2,média,Com R3/R4,3.69,REAL,ESTIMATIVA DO MODELO
2,42,Jaraguá,Pirituba/Jaraguá,211610.0,37663,1.162,18,14587,2816,1865,18110.0,NaN,NaN,177.983082,NaN,0.085582,NaN,0.387303,0.155494,"CRAS - Rua Barra Da Forquilha, 300",0.167902,0.393675,225.0,240.0,825.0,18.5,média,Com R3/R4,3.90,REAL,ESTIMATIVA DO MODELO
3,11,Brasilândia,Freguesia/Brasilândia,243273.0,53409,0.217,52,21366,4339,3199,22185.0,NaN,NaN,219.543476,NaN,0.091194,NaN,0.400045,0.195583,"CRAS - Rua Ibiraiaras, 372",0.252195,0.411541,1256.0,614.0,2791.0,34.9,urgente,Com R3/R4,11.47,REAL,ESTIMATIVA DO MODELO
4,29,Freguesia do Ó,Freguesia/Brasilândia,137240.0,16880,0.781,5,5460,2067,905,20781.0,NaN,NaN,122.996211,NaN,0.151421,NaN,0.323460,0.099466,"CRAS - Avenida Itaberaba, 924",0.091803,0.361568,0.0,643.0,1607.5,15.9,média,Com R3/R4,11.71,REAL,ESTIMATIVA DO MODELO
